# 01 -- Paragraph index and data sanity checks

Builds the paragraph segmentation of paper §III-C and asserts the counts
reported there. **Run this first**: if it fails, nothing downstream is valid.

Segmentation rule (matches the detectors' own preprocessing):
`content.splitlines()` -> drop blank lines -> drop lines longer than 1,000
characters.

| | Articles | Paragraphs | Gold spans |
|---|---:|---:|---:|
| EN | 90 | 3,127 | 1,801 |
| PL | 49 | 779 | 985 |
| RU | 48 | 503 | 739 |

> **Polish gold**: the CheckThat! 2024 PL dev span file ships with every row
> duplicated (1,970 rows = 985 unique x 2). `load_spans` de-duplicates on
> `(article_id, technique, start, end)`. All PL numbers in the paper use the
> de-duplicated gold.

In [ ]:
import sys; sys.path.insert(0, 'src')
from common import *

print(f'{C} techniques loaded')
print(f'articles dir : {ARTICLES}')
print(f'gold dir     : {GOLD_DIR}')
print(f'predictions  : {PRED_DIR}')

In [ ]:
# --- paragraph index ---
PARA_IDX = {l: build_para_index(l) for l in LANGS}

print(f"{'lang':<5} {'articles':>9} {'expected':>9} {'paragraphs':>11} {'expected':>9}")
print('-' * 48)
for l in LANGS:
    na = len(PARA_IDX[l])
    npar = sum(len(v) for v in PARA_IDX[l].values())
    print(f'{LANG_LABEL[l]:<5} {na:>9} {EXPECT_ARTICLES[l]:>9} '
          f'{npar:>11} {EXPECT_PARAGRAPHS[l]:>9}')
    assert na == EXPECT_ARTICLES[l], f'{l}: {na} articles, expected {EXPECT_ARTICLES[l]}'
    assert npar == EXPECT_PARAGRAPHS[l], f'{l}: {npar} paragraphs, expected {EXPECT_PARAGRAPHS[l]}'
print('\nOK -- paragraph segmentation matches paper §III-C')

In [ ]:
# --- gold spans (PL de-duplicated) ---
GOLD = {}
for l in LANGS:
    raw = load_spans(gold_path(l), l, dedup=False)
    ded = load_spans(gold_path(l), l, dedup=True)
    GOLD[l] = ded
    flag = ' (de-duplicated 2x)' if len(raw) == 2 * len(ded) else ''
    print(f'{LANG_LABEL[l]}: {len(raw):>5} rows -> {len(ded):>5} unique spans '
          f'(expected {EXPECT_GOLD_SPANS[l]}){flag}')
    assert len(ded) == EXPECT_GOLD_SPANS[l], f'{l}: {len(ded)} spans'
print('\nOK -- gold span counts match paper §III-C')

In [ ]:
# --- gold spans -> paragraphs ---
GOLD_PS = {}
for l in LANGS:
    ps, n_un = spans_to_paragraphs(GOLD[l], PARA_IDX[l])
    GOLD_PS[l] = ps
    pct = 100 * n_un / len(GOLD[l])
    print(f'{LANG_LABEL[l]}: {len(ps):>5} paragraphs carry gold | '
          f'{n_un:>3} spans unreachable ({pct:.1f}%)')

print('\nPaper §III-C reports 11.9% (PL) and 8.7% (RU) of gold spans')
print('discarded by the len<=1000 filter -- these are unreachable by the')
print('detectors themselves, since the detectors apply the same filter.')

In [ ]:
# --- prediction files present and well-formed? ---
print(f"{'config':<18} {'rows':>7} {'articles':>9} {'techniques':>11}")
print('-' * 48)
for l, m in CONFIGS:
    p = pred_path(l, m)
    assert p.exists(), f'missing: {p}'
    df = load_pred(l, m)
    print(f'{LANG_LABEL[l]} {METHOD_LABEL[m]:<12} {len(df):>7} '
          f'{df["article_id"].nunique():>9} {df["technique"].nunique():>11}')
print('\nOK -- all nine prediction files readable')